# InsureIQ — Colab Runner

Clones [`aksri648/INSURE-IQ`](https://github.com/aksri648/INSURE-IQ), installs Ollama + Python deps, pulls VRAM-sized models, and exposes the Streamlit app on a public `trycloudflare.com` URL.

**Runtime:** `Runtime → Change runtime type → GPU` (T4 free works; A100 better).

Run cells top to bottom. The last cell prints the public URL.

## Cell 1 — GPU check

In [ ]:
import subprocess

try:
    gpu = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
         "--format=csv,noheader"],
        capture_output=True, text=True, check=True
    ).stdout.strip()
    print("\U0001f5a5  GPU:\n" + gpu)
except Exception as e:
    print("\u26a0\ufe0f  No GPU detected — enable one via Runtime menu. Detail:", e)

print("\n\U0001f4be RAM:")
print(subprocess.run(["free", "-h"], capture_output=True, text=True).stdout)

## Cell 2 — Clone the repo

In [ ]:
%%bash
set -e
cd /content
rm -rf INSURE-IQ
git clone --depth 1 https://github.com/aksri648/INSURE-IQ.git
ls INSURE-IQ

## Cell 3 — Install system + Python dependencies

In [ ]:
%%bash
set -e
apt-get install -qq -y curl libgl1 libglib2.0-0 > /dev/null
pip install -q -r /content/INSURE-IQ/requirements.txt
echo "\u2705 Dependencies installed"

## Cell 4 — Install + start Ollama

In [ ]:
%%bash
set -e
if ! command -v ollama >/dev/null 2>&1; then
    echo "\u2b07\ufe0f  Installing Ollama..."
    curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
fi

echo "\U0001f680 Starting Ollama in background..."
nohup ollama serve > /tmp/ollama.log 2>&1 &

for i in {1..15}; do
    if curl -s http://localhost:11434/api/tags > /dev/null 2>&1; then
        echo "\u2705 Ollama running on :11434"
        exit 0
    fi
    sleep 2
done
echo "\u26a0\ufe0f  Ollama did not respond in time. Tail of log:"
tail -20 /tmp/ollama.log

## Cell 5 — Pull models (sized to detected VRAM)

In [ ]:
%%bash
set -e
VRAM=$(nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits | head -1)
echo "\U0001f50d Detected VRAM: ${VRAM} MiB"

if [ "$VRAM" -gt 35000 ]; then
    OCR_MODEL="llava:13b"
    ANALYST_MODEL="deepseek-r1:14b"
else
    OCR_MODEL="llava:7b"
    ANALYST_MODEL="deepseek-r1:7b"
fi
EMBED_MODEL="nomic-embed-text"

echo "\U0001f4e5 Pulling $EMBED_MODEL"
ollama pull $EMBED_MODEL
echo "\U0001f4e5 Pulling $OCR_MODEL (may take several minutes)"
ollama pull $OCR_MODEL
echo "\U0001f4e5 Pulling $ANALYST_MODEL (may take several minutes)"
ollama pull $ANALYST_MODEL

cat > /tmp/model_config.env <<EOF
OCR_MODEL=$OCR_MODEL
ANALYST_MODEL=$ANALYST_MODEL
EMBED_MODEL=$EMBED_MODEL
EOF
echo "\u2705 Models ready. Config written to /tmp/model_config.env"
cat /tmp/model_config.env

## Cell 6 — Set the Tavily API key (optional)

Leave the value blank to skip external research. Get a free key at <https://tavily.com>.

In [ ]:
import os

os.environ["TAVILY_API_KEY"] = ""  # ← paste your tvly-... key here, or leave blank

# Persist for the streamlit subprocess via the project .env
env_path = "/content/INSURE-IQ/.env"
with open(env_path, "w") as f:
    f.write(f"TAVILY_API_KEY={os.environ['TAVILY_API_KEY']}\n")

print("\u2705 .env written:", env_path)

## Cell 7 — Install cloudflared

In [ ]:
%%bash
set -e
if ! command -v cloudflared >/dev/null 2>&1; then
    wget -q -O /usr/local/bin/cloudflared \
        https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    chmod +x /usr/local/bin/cloudflared
fi
cloudflared --version

## Cell 8 — Launch Streamlit + Cloudflare tunnel

Prints the public HTTPS URL. Keep this cell running for the duration of the session.

In [ ]:
import os
import re
import subprocess
import threading
import time

import requests

PROJECT_DIR = "/content/INSURE-IQ"

# 1. Start Streamlit
def start_streamlit():
    env = os.environ.copy()
    env["PYTHONPATH"] = PROJECT_DIR
    subprocess.Popen(
        [
            "streamlit", "run", f"{PROJECT_DIR}/app.py",
            "--server.port", "8501",
            "--server.headless", "true",
            "--server.enableCORS", "false",
            "--server.enableXsrfProtection", "false",
            "--browser.gatherUsageStats", "false",
        ],
        cwd=PROJECT_DIR,
        env=env,
        stdout=open("/tmp/streamlit.log", "wb"),
        stderr=subprocess.STDOUT,
    )

print("\U0001f680 Starting Streamlit...")
threading.Thread(target=start_streamlit, daemon=True).start()

# Wait until Streamlit answers
for i in range(20):
    try:
        if requests.get("http://localhost:8501", timeout=2).status_code < 500:
            print("\u2705 Streamlit is up on :8501")
            break
    except Exception:
        time.sleep(2)
else:
    print("\u26a0\ufe0f  Streamlit slow to start. Tail of log:")
    subprocess.run(["tail", "-30", "/tmp/streamlit.log"])

# 2. Start cloudflared and parse the public URL
print("\U0001f310 Opening Cloudflare tunnel...")
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--no-autoupdate",
     "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

url_pattern = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
public_url = None
for line in tunnel.stdout:
    print(line.rstrip())
    m = url_pattern.search(line)
    if m:
        public_url = m.group(0)
        print("\n" + "=" * 64)
        print(f"  \U0001f310 PUBLIC URL: {public_url}")
        print("=" * 64)
        print("  Share this link \u2014 valid while this Colab session is running.")
        print("=" * 64 + "\n")
        break

if not public_url:
    raise RuntimeError("cloudflared did not print a public URL. Check the log above.")

# Keep the cell alive so the tunnel stays open
print("\u23f3 Tunnel is open. Leave this cell running. Interrupt to stop.")
try:
    for line in tunnel.stdout:
        print(line.rstrip())
except KeyboardInterrupt:
    tunnel.terminate()
    print("\n\U0001f6d1 Tunnel stopped.")

## (Optional) Keep-alive — prevent Colab idle timeout

Run **before** Cell 8 if you expect long analyses on the free tier.

In [ ]:
import IPython

IPython.display.display(IPython.display.Javascript("""
function ClickConnect(){
    console.log("Keeping Colab alive...");
    const btn = document.querySelector("#top-toolbar > colab-connect-button");
    if (btn && btn.shadowRoot) {
        const inner = btn.shadowRoot.querySelector("#connect");
        if (inner) inner.click();
    }
}
setInterval(ClickConnect, 60000);
"""))
print("\u2705 Keep-alive activated (clicks Connect every 60s).")